In [21]:
import os
import pickle
import numpy as np
import pandas as pd
import librosa
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tqdm import tqdm

# Paths
audio_dir       = '../audio_clips/UI_Clips/'     # folder containing .wav files
RFmodel_path    = './RandomForest.pkl'
SVMmodel_path   = './SVM.pkl'
output_csv      = './batch_predictions.csv'

# Load classifiers
with open(RFmodel_path, 'rb') as f:
    clfRF = pickle.load(f)
with open(SVMmodel_path, 'rb') as f:
    clfSVM = pickle.load(f)

# Load Wav2Vec2 processor and model
processor     = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec_model.eval()

# Feature extraction function
def extract_wave2vec_features(path):
    try:
        audio, sr = librosa.load(path, sr=16000)
        inputs    = processor(audio, sampling_rate=16000, return_tensors="pt", padding=True)
        with torch.no_grad():
            outputs = wav2vec_model(**inputs)
        return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    except Exception as e:
        print(f"Error on {path}: {e}")
        return np.zeros(768)

# Loop over all .wav files and collect predictions
results = []
for fname in tqdm(os.listdir(audio_dir)):
    if not fname.lower().endswith('.wav'):
        continue

    full_path = os.path.join(audio_dir, fname)
    feat       = extract_wave2vec_features(full_path).reshape(1, -1)

    # Model outputs
    rf_pred  = clfRF.predict(feat)[0]
    svm_pred = clfSVM.predict(feat)[0]
    rf_prob  = clfRF.predict_proba(feat)[0] if hasattr(clfRF, "predict_proba") else [None, None]
    svm_prob = clfSVM.predict_proba(feat)[0] if hasattr(clfSVM, "predict_proba") else [None, None]

    results.append({
        "filename": fname,
        "RF_label": rf_pred,
        "RF_prob_deceptive": rf_prob[0],
        "RF_prob_truthful": rf_prob[1],
        "SVM_label": svm_pred,
        "SVM_prob_deceptive": svm_prob[0],
        "SVM_prob_truthful": svm_prob[1],
    })

# Save to CSV
df = pd.DataFrame(results)
df.to_csv(output_csv, index=False)
print(f"[INFO] Batch predictions saved to {output_csv}")


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:08<00:00,  1.20it/s]

[INFO] Batch predictions saved to ./batch_predictions.csv
